# NZZ ContextAI — RAG Pipeline Test

In [8]:
import sys
sys.path.insert(0, "../src")

from config import (
    CHROMA_PATH, CHROMA_COLLECTION,
    EMBEDDING_MODEL, RERANKER_MODEL, USE_RERANKING,
    EVAL_TOP_K_RETRIEVAL, EVAL_TOP_K_FINAL,
)
from embed import get_chroma_collection
from retrieval import load_models, retrieve
from generate import generate

print("Imports OK")

Imports OK


## 1 — Modelle & Datenbank laden

In [9]:
collection = get_chroma_collection(CHROMA_PATH, CHROMA_COLLECTION)
model, reranker = load_models(use_reranking=USE_RERANKING)

print(f"Chunks in ChromaDB: {collection.count():,}")
print(f"Embedding-Modell:   {EMBEDDING_MODEL}")
print(f"Reranker:           {RERANKER_MODEL if USE_RERANKING else 'deaktiviert'}")

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: cross-encoder/mmarco-mMiniLMv2-L12-H384-v1
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chunks in ChromaDB: 3,140
Embedding-Modell:   Qwen/Qwen3-Embedding-0.6B
Reranker:           cross-encoder/mmarco-mMiniLMv2-L12-H384-v1


## 2 — Einzelne Anfrage testen

In [10]:
QUERY = "Wie werden ukrainische Kinder in Russland umerzogen?"

In [11]:
chunks = retrieve(
    QUERY, model, collection, reranker,
    top_k_retrieval=EVAL_TOP_K_RETRIEVAL,
    top_k_rerank=EVAL_TOP_K_FINAL,
)

print(f"Top-{len(chunks)} Quellen fuer: '{QUERY}'\n")
for i, c in enumerate(chunks, 1):
    score = c.get("rerank_score", c.get("similarity_score", 0))
    print(f"  [{i}] {c['title'][:65]}")
    print(f"       Artikel-ID: {c['article_id']}  |  Datum: {c.get('published_date', '-')}  |  Score: {score:.3f}")

Top-5 Quellen fuer: 'Wie werden ukrainische Kinder in Russland umerzogen?'

  [1] Die stummen Opfer des Krieges: wie Russland ukrainische Kinder zu
       Artikel-ID: 336520888  |  Datum: 2025-12-13  |  Score: 8.188
  [2] Ukrainische Kinder in Russland aufgespürt: Wie KI bei der Suche h
       Artikel-ID: 336520876  |  Datum: 2025-12-13  |  Score: 6.422
  [3] Die stummen Opfer des Krieges: wie Russland ukrainische Kinder zu
       Artikel-ID: 336520888  |  Datum: 2025-12-13  |  Score: 5.426
  [4] Die stummen Opfer des Krieges: wie Russland ukrainische Kinder zu
       Artikel-ID: 336520888  |  Datum: 2025-12-13  |  Score: 5.176
  [5] Die stummen Opfer des Krieges: wie Russland ukrainische Kinder zu
       Artikel-ID: 336520888  |  Datum: 2025-12-13  |  Score: 4.416


## 3 — Antwort generieren

In [12]:
answer = generate(QUERY, chunks)
print(answer)

**Direkte Antwort**
Russland umerzieht ukrainische Kinder als Teil seiner nationalen Propaganda und um sie zu patriotischen Russen zu machen. Diese Umerziehung erfolgt oft durch Zwang, da die Kinder ihre ukrainische Identität verloren und sich stattdessen als Russen identifizieren müssen.

**Kontextualisierung**
Die Umerziehung der ukrainischen Kinder in Russland ist ein Teil des Krieges und seiner Auswirkungen auf das zivile Leben. Die Kinder werden oft von ihren Eltern getrennt, um sie in russische Familien oder Waisenhäuser zu bringen, wo sie unter Zwang eine neue Identität annehmen müssen. Dies geschieht oft ohne die Zustimmung der Eltern und kann als Kriegsverbrechen angesehen werden.

**Quellenübersicht**
Die Quellen für diese Informationen sind:

* [1] Artikel-ID: 336520888 | Titel: Die stummen Opfer des Krieges: wie Russland ukrainische Kinder zu patriotischen Russen umerzieht | Datum: 2025-12-13 | Ressort: International
* [2] Artikel-ID: 336520876 | Titel: Ukrainische Kinder i

## 4 — Mehrere Ground-Truth-Queries durchlaufen

In [13]:
import json
from config import EVAL_GROUND_TRUTH

with open(EVAL_GROUND_TRUTH) as f:
    ground_truth = [json.loads(line) for line in f if line.strip()]

print(f"{len(ground_truth)} Queries geladen")

14 Queries geladen


In [14]:
N = 3

for sample in ground_truth[:N]:
    query = sample["query"]
    relevant = set(sample["relevant_article_ids"])

    results = retrieve(
        query, model, collection, reranker,
        top_k_retrieval=EVAL_TOP_K_RETRIEVAL,
        top_k_rerank=EVAL_TOP_K_FINAL,
    )

    top1_id = results[0]["article_id"] if results else ""
    hit = "HIT" if top1_id in relevant else "MISS"

    print(f"[{hit}] {query}")
    print(f"  Erwartet: {relevant}")
    print(f"  Top-1:    {top1_id} — {results[0]['title'][:60] if results else '-'}")

    answer = generate(query, results)
    print(f"  Antwort:  {answer}")
    print()

[HIT] Was erwarten Schweizer Anlageprofis für die Börsenmärkte im Jahr 2026?
  Erwartet: {'336541061'}
  Top-1:    336541061 — «Wir werden in spätestens 18 Monaten wieder Negativzinsen ha
  Antwort:  **Direkte Antwort**
Die Schweizer Wirtschaft erwarte im Jahr 2026 ein reales Wachstum des Bruttoinlandprodukts (BIP) von 1,1 Prozent, wie das Staatssekretariat für Wirtschaft und das KOF-Institut der ETH Zürich mitteilten. Die Zollvereinbarung mit den USA hat die Aussichten für die Schweizer Wirtschaft etwas aufgehellt.

**Kontextualisierung**
Die Schweiz erwarte im Jahr 2026 ein reales Wachstum des BIP von 1,1 Prozent, was bedeutet, dass das Land trotz der Zollvereinbarung mit den USA und der anziehenden Konjunktur in Europa weiterhin unterdurchschnittliches Wirtschaftswachstum erwarten wird. Die Schweizer Industrie erwarte im ersten Halbjahr 2026 dank den Impulsen aus den USA und der anziehenden Konjunktur in Europa deutlich auszuweiten, aber das würde nicht unbedingt zu einer Stabilisie